# 第07回 ソルバ1：Pythonから線形計画を解く

今日の目標は、前回の工場の生産計画を自分のColabで解き、結果を確かめることです。小レポート・提出はありません。

## 1. 自分のノートブックを用意する

1. Googleアカウントでログインし、「ファイル → ドライブにコピーを保存」を選びます。
2. 自分のDriveに保存したコピーで作業します。
3. コード左の▶か Shift+Enter で、上から順に実行します。

ノートブックは説明のセルとコードのセルからなります。コードの実行場所はGoogle側の計算機です。CPUで十分です。最初の接続には少し時間がかかります。

In [ ]:
1 + 1

上の式を `1 + 3` に変更して再実行してください。結果が4になったら、変更と実行ができています。

## 2. 変数・数値・文字列

`=` は代入です。右側の値に、左側の名前を付けます。文字列は半角の引用符で囲み、`print` で値を表示します。

In [ ]:
product = "製品1"
unit_profit = 4
quantity = 3
profit = unit_profit * quantity
print(product, profit, "万円")

`quantity` はPythonで値を保持する名前です。後でPuLPで作る決定変数は「これからソルバが値を決めるもの」で、役割が違います。

`*` は掛け算、`/` は割り算、`**` は累乗です。数式の掛け算記号を省略して `4x1` と書くことはできません。

### やってみよう1
単位利益5万円の製品を2.5単位作るときの利益を計算してください。下の値を変更し、実行します。

In [ ]:
unit_profit = 4  # 5へ変更
quantity = 3     # 2.5へ変更
profit = unit_profit * quantity
print(profit)

### 解答1
`unit_profit = 5`、`quantity = 2.5` とすれば、`5 * 2.5 = 12.5` 万円です。

## 3. リストと添字

複数の値は `[]` の中にカンマで並べます。添字は0から始まります。

In [ ]:
profits = [4, 5]
amounts = [3, 1]
print(profits[0])
print(profits[1])
print(profits[0]*amounts[0] + profits[1]*amounts[1])

表示は4、5、17です。`profits[2]` は存在せず、IndexErrorになります。数学の製品1・2を、Pythonでは添字0・1で扱っています。

## 4. forと字下げ

`for v in values:` で、各要素を順に `v` として使います。繰り返す行は半角スペース4個で字下げします。

In [ ]:
values = [4, 5, 6]
total = 0
for v in values:
    total = total + v
print(total)

`total = 0` は繰り返しの前に置きます。ループ内に置くと、毎回それまでの合計を消してしまいます。最後の表示は15です。

### やってみよう2
`values = [2, 3, 7]` の合計をforで求めてください。初期化と字下げに注意します。

In [ ]:
values = [2, 3, 7]
# 下に初期化、for、printを書いて実行してください。

### 解答2

In [ ]:
total = 0
for v in values:
    total = total + v
print(total)  # 12

## 5. 実行順と保存

セルの値は、画面の順ではなく実行した順に変わります。途中の値を変更したら、それを使う後のセルも再実行してください。

ノートブックの保存と、実行中の変数の保存は別です。接続が切れ、新しい実行環境になったら上から実行し直します。まず保存表示を確認し、必要なら Ctrl+S / Cmd+S で保存します。

## 6. PuLPとソルバを準備する

PuLPは目的関数・制約をPythonで記述するライブラリです。実際の計算はソルバに渡します。この教材はHiGHSを使います。前年度のCBCとはソルバが異なりますが、表す線形計画は同じです。

以下はColabで実行するインストール用セルです。授業の再現性のため版を固定しています。新しい実行環境では再実行してください。

In [ ]:
%pip -q install "pulp==3.3.0" "highspy==1.12.0"

In [ ]:
import pulp
print("PuLP", pulp.__version__)
print("利用可能なソルバ:", pulp.listSolvers(onlyAvailable=True))

## 7. 前回の工場をモデルにする

製品1・2の生産量 $x_1,x_2$ は連続量、利益は万円です。

$$\begin{aligned}\max\quad &4x_1+5x_2\\\text{制約}\quad &x_1+2x_2\le5\\&2x_1+2x_2\le8\\&x_1+x_2\le10\\&x_1,x_2\ge0\end{aligned}$$

原料A・B・Cの在庫が5・8・10kgです。`LpMaximize` はPuLPの定数、`lowBound=0` は非負制約、`None` はその側の境界を指定しないことを表します。

In [ ]:
model = pulp.LpProblem("Production", pulp.LpMaximize)
x1 = pulp.LpVariable("x1", lowBound=0)
x2 = pulp.LpVariable("x2", lowBound=0)
model += 4*x1 + 5*x2, "profit"
model += x1 + 2*x2 <= 5, "material_A"
model += 2*x1 + 2*x2 <= 8, "material_B"
model += x1 + x2 <= 10, "material_C"
print(model)

`model += 式` で目的関数を、`model += 不等式` で制約を追加します。これはPuLPが用意した書き方です。目的関数は1つだけ指定します。

等式制約は `==`、代入は `=`。不等号は `<=`・`>=` です。ソルバへ渡すために、必ずスラック変数を自分で追加する必要はありません。

## 8. 解く前後の確認

先ほど出力したモデルが数式と一致することを確かめてから、解きます。`if` の字下げされた行は条件が成立するときだけ実行します。

In [ ]:
model.solve(pulp.HiGHS(msg=False))
status = pulp.LpStatus[model.status]
print("状態:", status)
if status == "Optimal":
    print("製品1:", pulp.value(x1))
    print("製品2:", pulp.value(x2))
    print("利益（万円）:", pulp.value(model.objective))
else:
    print("最適解として読める状態ではありません。条件とログを確認してください。")

得られるのは $x_1=3, x_2=1$、利益17万円です。`Optimal` は最適解が得られた状態です。`Infeasible` は実行可能解がない、`Unbounded` は目的値を際限なく改善できる状態です。`Not Solved`・`Undefined` だけから「問題が難しい」「解がない」と断定しません。

In [ ]:
if status == "Optimal":
    a, b = pulp.value(x1), pulp.value(x2)
    print("原料A:", a + 2*b, "<= 5")
    print("原料B:", 2*a + 2*b, "<= 8")
    print("原料C:", a + b, "<= 10")
    print("利益:", 4*a + 5*b)

### やってみよう3
原料Aだけを6kgに変更してください。下のセルの右辺5を6に書き換えて実行します。他の条件は変えません。

In [ ]:
trial = pulp.LpProblem("Production", pulp.LpMaximize)
u1 = pulp.LpVariable("u1", lowBound=0)
u2 = pulp.LpVariable("u2", lowBound=0)
trial += 4*u1 + 5*u2, "profit"
trial += u1 + 2*u2 <= 5, "material_A"
trial += 2*u1 + 2*u2 <= 8, "material_B"
trial += u1 + u2 <= 10, "material_C"
trial.solve(pulp.HiGHS(msg=False))
if pulp.LpStatus[trial.status] == "Optimal":
    print(pulp.value(u1), pulp.value(u2), pulp.value(trial.objective))

### 解答3
$u_1+2u_2\le6$ に直すと、$u_1=2,u_2=2$、利益18万円です。原料Bは8kg、Cは4kg使用します。原料Aを1kg増やすと、この範囲では利益が1万円増えます。06で扱った限界価値ともつながります。

## 9. よくあるエラーを直す

| 表示・症状 | 確認すること |
|---|---|
| NameError | 名前の綴り、値を定義したセルを実行したか |
| ModuleNotFoundError | インストール用セルとimportを実行したか |
| IndentationError | for・ifの字下げが揃っているか |
| 結果が変わらない | 編集したセルと、その後の計算を再実行したか |

エラー表示の最後の行と、示されたコードの行から確認します。

## まとめと次回

数式 → PuLPのモデル → ソルバ → 状態の確認 → 元の条件で検算、の順です。保存したノートブックを開き直せることを確認してください。次回も端末を使い、表のデータを扱って輸送問題を解きます。

参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html)、[PuLP](https://github.com/coin-or/pulp/tree/3.3.0)、[HiGHS](https://highs.dev/)。